# ukgeo — UK Geocoding Reference Data

This notebook demonstrates the [ukgeo](https://github.com/ThomasHSimm/ukgeo) Python geocoder
using the pre-built dataset hosted here on Kaggle.

ukgeo resolves messy UK location text — road references, junction names, place names,
colloquial names, old county names — to latitude/longitude coordinates. It is designed
for bulk offline processing without API calls or rate limits.

**Use cases:**
- Road safety analysis with STATS19 collision data
- Transport research requiring UK road network awareness
- Any application where UK location text needs geocoding at scale


In [ ]:
# Install ukgeo from GitHub (PyPI coming soon)
!pip install git+https://github.com/ThomasHSimm/ukgeo.git -q

import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import shutil
import ukgeo

# Copy Kaggle dataset to the data path ukgeo auto-detects.
source_path = Path("/kaggle/input/ukgeo-combined-dataset/ukgeo_data.parquet")
package_root = Path(ukgeo.__file__).resolve().parents[1]
dataset_path = package_root / "data" / "kaggle" / "ukgeo_data.parquet"
dataset_path.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(source_path, dataset_path)

from ukgeo import Geocoder
geo = Geocoder()
print("Geocoder ready")


## Basic usage — single queries

ukgeo handles a wide range of UK location input types.


In [ ]:
test_cases = [
    ("Full postcode",           "LS1 1BA"),
    ("Motorway junction",       "M62 Junction 26"),
    ("A-road + place",          "A647 near Bradford"),
    ("Named interchange",       "Spaghetti Junction Birmingham"),
    ("Colloquial roundabout",   "Magic Roundabout Swindon"),
    ("Place + county",          "Skipton, North Yorkshire"),
    ("Typo",                    "Brafford West Yorkshire"),
    ("B-road + place",          "B6265 Bradford"),
    ("Old county name",         "Bournemouth Dorset"),
]

for label, text in test_cases:
    r = geo.geocode(text)
    status = f"{r.lat:.4f}, {r.lon:.4f}" if r.resolved else "unresolved"
    print(f"{label:<30} {text:<40} → {status} ({r.confidence})")


**Note:** "Lofthouse Interchange" resolves to the nearby Lofthouse Gate suburb rather
than the M62/M1 interchange itself — named interchange geometry is a known limitation.
See [docs/gaps_and_ecosystem.md](https://github.com/ThomasHSimm/ukgeo/blob/main/docs/gaps_and_ecosystem.md).

---

## Bulk geocoding

`geocode_batch()` processes a list or polars Series efficiently.
All OS data is loaded once at startup — subsequent queries are fast in-process lookups.


In [ ]:
locations = [
    "M1 Junction 42",
    "A64 York bypass near Tadcaster",
    "Dartford Crossing Kent",
    "Station Road Leeds",
    "Aberford West Yorkshire",
    "B1224 York",
    "Lofthouse Interchange",
    "WF10 4QH",
    "A1(M) Junction 47 Garforth",
    "Sighthill Edinburgh",
]

results = geo.geocode_batch(locations, show_progress=False)
print(results.select(["input", "lat", "lon", "confidence", "level_resolved", "interpreted_as"]))


## Performance benchmark

Benchmarked against 5,000 STATS19 2024 collision records — real road reference inputs
with verified ground truth coordinates.


In [ ]:
benchmark_data = {
    "Metric": ["Resolve rate", "Median error", "Within 500m", "Within 5km", "B-roads resolved"],
    "v0.1": ["—", "—", "—", "—", "—"],
    "v0.2": ["68.9%", "4,593m", "9.7%", "51.6%", "5.1%"],
    "v0.3": ["99.9%", "3,299m", "~10%", "59.9%", "99.9%"],
}
df = pd.DataFrame(benchmark_data)
print(df.to_string(index=False))
print("\nNote: median error reflects road centroid resolution.")
print("STATS19 records already have GPS coordinates — ukgeo is most useful")
print("for derived datasets that have road references but no coordinates.")


## What's in the dataset


In [ ]:
data = pl.read_parquet(dataset_path)

print(f"Total rows: {len(data):,}")
print(f"Built:      {data['BUILT_AT'][0]}")
print()
print("By source:")
print(data.group_by("SOURCE").agg(pl.len().alias("rows")).sort("rows", descending=True))
print()
print("By LOCAL_TYPE (top 15):")
print(data.group_by("LOCAL_TYPE").agg(pl.len().alias("rows")).sort("rows", descending=True).head(15))


## Further reading

- [ukgeo GitHub repo](https://github.com/ThomasHSimm/ukgeo) — source code, documentation, download scripts
- [Open Road Risk](https://github.com/ThomasHSimm/open-road-risk) — road safety risk modelling pipeline
- [docs/alternative.md](https://github.com/ThomasHSimm/ukgeo/blob/main/docs/alternative.md) — comparison with other UK geocoding tools
- [docs/STATUS.md](https://github.com/ThomasHSimm/ukgeo/blob/main/docs/STATUS.md) — current test results

**Data sources:**
- OS Open Names © Crown Copyright and database right 2024 (OGL v3)
- OS Open Roads © Crown Copyright and database right 2024 (OGL v3)
- © OpenStreetMap contributors (ODbL v1.0)
